In [ ]:
import pandas as pd
import sqlite3
import os

# Step 1: Read CSV file
file_path = "sales.csv"

if not os.path.exists(file_path):
    data = pd.DataFrame({
        "id": [1, 2, 3, 4, 5],
        "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Printer"],
        "amount": [50000, 800, 1500, 12000, 7000]
    })
    data.to_csv(file_path, index=False)

df = pd.read_csv(file_path)

# Step 2: Connect to database
conn = sqlite3.connect("sales.db")

# Step 3: Create target table if it does not exist
conn.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY,
    product TEXT,
    amount REAL
)
""")

# Step 4: Find the last loaded ID
result = pd.read_sql(
    "SELECT MAX(id) AS last_id FROM sales",
    conn
)

last_id = result["last_id"].iloc[0]

if pd.isna(last_id):
    last_id = 0

# Step 5: Select only new records
new_data = df[df["id"] > last_id]

# Step 6: Load new records
if not new_data.empty:
    new_data.to_sql(
        "sales",
        conn,
        if_exists="append",
        index=False
    )
    print("New records loaded successfully.")
else:
    print("No new records found.")

# Step 7: Display final data
final_data = pd.read_sql("SELECT * FROM sales", conn)

print("\nData in Database:")
print(final_data)

conn.close()

New records loaded successfully.

Data in Database:
   id   product   amount
0   1    Laptop  50000.0
1   2     Mouse    800.0
2   3  Keyboard   1500.0
3   4   Monitor  12000.0
4   5   Printer   7000.0


Result
The ETL pipeline successfully performs incremental loading by identifying the last loaded record and inserting only new records into the database. This avoids reloading the complete dataset and prevents duplicate records.